# Inspect ChromaDB Chunks

This notebook loads your local ChromaDB collection and prints chunk text + metadata.


In [1]:
import chromadb
import pandas as pd
from pathlib import Path
import json

# Use the same Chroma persist dir as the backend.
PROJECT_ROOT = Path("/Users/ajeyds/Projects/Doc Gap Analysis")
PERSIST_DIR = str(PROJECT_ROOT / "data/chroma_db")
COLLECTION_NAME = "rag_chunks"

# Filter to a single uploaded file.
# Default: auto-pick the latest entry from data/kb_metadata.json.
SOURCE_PATH = None

try:
    meta_path = PROJECT_ROOT / "data/kb_metadata.json"
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    latest_entry = max(meta["files"].values(), key=lambda e: e.get("uploadedAt", ""))
    SOURCE_PATH = latest_entry.get("path")
except Exception as e:
    print("Could not auto-resolve SOURCE_PATH:", e)


In [2]:
# 1) Connect to ChromaDB
client = chromadb.PersistentClient(path=PERSIST_DIR)
collection = client.get_collection(name=COLLECTION_NAME)

print("Notebook cwd:", Path.cwd())
print("Using PERSIST_DIR:", PERSIST_DIR)
print("Collection:", COLLECTION_NAME)
print("Total items:", collection.count())


Notebook cwd: /Users/ajeyds/Projects/Doc Gap Analysis
Using PERSIST_DIR: /Users/ajeyds/Projects/Doc Gap Analysis/data/chroma_db
Collection: rag_chunks
Total items: 69


In [3]:
# 2) Retrieve data
# NOTE: We intentionally do NOT include embeddings to keep the output small.
include = ["documents", "metadatas"]

if SOURCE_PATH:
    data = collection.get(where={"source": SOURCE_PATH}, include=include)
else:
    # Pull only the first N for quick inspection.
    # If you want everything, increase or remove the limit.
    data = collection.get(include=include, limit=200)

ids = data.get("ids", [])
documents = data.get("documents", [])
metadatas = data.get("metadatas", [])

print("Fetched rows:", len(ids))
print("Example metadata keys:", sorted((metadatas[0] or {}).keys()) if metadatas else [])


Fetched rows: 21
Example metadata keys: ['chunk_id', 'chunk_type', 'date', 'document_summary', 'document_title', 'document_type', 'domain', 'gap_type', 'parent_context', 'source', 'total_acceptance_criteria', 'total_stories', 'version']


In [4]:
# 3) Format into a table
rows = []
for _id, doc, meta in zip(ids, documents, metadatas):
    meta = meta or {}
    rows.append(
        {
            "id": _id,
            "chunk_type": meta.get("chunk_type"),
            "chunk_id": meta.get("chunk_id"),
            "parent_context": meta.get("parent_context"),
            "source": meta.get("source"),
            "document": doc,
        }
    )

df = pd.DataFrame(rows, columns=["id", "chunk_type", "chunk_id", "parent_context", "source", "document"])
df = df.sort_values(by=["chunk_type", "chunk_id"], na_position="last")

# Show a preview (no embeddings)
df.head(30)


,id,chunk_type,chunk_id,parent_context,source,document
1,b07c8efca6fe9db670945f457a2bf3e4745b0ac642ecde...,ac,AC-1.1,Core Banking Sync,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Sync Interval — Sync runs every 4 hours.
2,fc25f3e2688bca00fa1b312c6b59c773f9abab0bdcdffe...,ac,AC-1.2,Core Banking Sync,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Missing Fields Handling — Missing fields resul...
3,b310519910b7bcb0bb848860f6267e13500f2c3318c6ec...,ac,AC-1.3,Core Banking Sync,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,"Duplicate Record Handling — On duplicate, upda..."
4,a1851c36ed34b508c96a63dcae22029f90662d5109cb45...,ac,AC-1.4,KYC Record Creation,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,KYC Record Requirements — Mandatory fields cap...
5,846ae60a7323803c50f9b88e312610fb6297d436d2701f...,ac,AC-1.5,KYC Record Creation,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Duplicate Blocking — Duplicate records are blo...
6,aeae0d3adb6a551ff11f28c5ed4f6afcf00441a2a19441...,ac,AC-1.6,KYC Record Creation,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Change Logging — Changes are logged with offic...
7,a65872d7a851b6ea8839e246494cbb07ef66ae93fa4db2...,ac,AC-2.1,Automated Credit Assessment Scheduling,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Credit Assessment Initiation — Assessment init...
8,62c6946330d237afe6c340f8d8f0821fd9a29d8735f94a...,ac,AC-2.2,Automated Credit Assessment Scheduling,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Reminder Scheduling — Reminders at 15 and 7 da...
9,3b9d44851e67d476fff2f3c0cbc3525275855e17b5284d...,ac,AC-2.3,Automated Credit Assessment Scheduling,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Overdue Status Setting — Overdue status set wh...
10,33934cca80fbd7b892fde8d6656fabef05348d6ad2ab69...,ac,AC-2.4,Loan Document Submission,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Document Submission Blockage — Missing documen...


In [6]:
print(df.loc[5, 'document'])

Duplicate Blocking — Duplicate records are blocked.


In [7]:
# 4) Print chunk text (optionally truncate)
N = 10
max_chars = 800  # set to None for full text

for i, row in df.head(N).iterrows():
    print("\n" + "=" * 80)
    print(f"[{row['chunk_type']}] chunk_id={row['chunk_id']}")
    print(f"parent_context={row['parent_context']}")
    print(f"source={row['source']}")
    text = row["document"] or ""
    if max_chars is not None:
        text = text[:max_chars] + ("..." if len(text) > max_chars else "")
    print(text)



[ac] chunk_id=AC-1.1
parent_context=Core Banking Sync
source=/Users/ajeyds/Projects/Doc Gap Analysis/data/uploads/kb/b2750661-b836-45f9-90e7-0cf42da5765d_v2_UserStories.docx
Sync Interval — Sync runs every 4 hours.

[ac] chunk_id=AC-1.2
parent_context=Core Banking Sync
source=/Users/ajeyds/Projects/Doc Gap Analysis/data/uploads/kb/b2750661-b836-45f9-90e7-0cf42da5765d_v2_UserStories.docx
Missing Fields Handling — Missing fields result in rejection and an error log.

[ac] chunk_id=AC-1.3
parent_context=Core Banking Sync
source=/Users/ajeyds/Projects/Doc Gap Analysis/data/uploads/kb/b2750661-b836-45f9-90e7-0cf42da5765d_v2_UserStories.docx
Duplicate Record Handling — On duplicate, update existing record.

[ac] chunk_id=AC-1.4
parent_context=KYC Record Creation
source=/Users/ajeyds/Projects/Doc Gap Analysis/data/uploads/kb/b2750661-b836-45f9-90e7-0cf42da5765d_v2_UserStories.docx
KYC Record Requirements — Mandatory fields captured; unique KYC ID generated.

[ac] chunk_id=AC-1.5
parent_conte